In [ ]:
import os
from pathlib import Path


# SwarmUI applies a theme by registering CSS paths (WebServer.RegisterTheme) and
# selecting one from the user Theme setting. The page layout falls back to
# modern_dark until that setting or the sui_theme_id cookie is set.
# Extra user CSS is an extension StyleSheetFiles entry, injected into the page
# header. Theme stylesheets are separate and load after that, so the layout
# sheet is also the last NECGRO theme file and wins over modern.css.
# The visible product name is hardcoded as "SwarmUI" in the layout title and
# version line. UserAuthorization.InstanceTitle is the instance name beside it.

NECGRO_THEME_CSS = """/* NECGRO theme colors. Dark and flat. */
:root {
    --background: #101114;
    --background-soft: #181a1f;
    --background-gray: #1c1e24;
    --background-gray-danger: #2a1818;
    --background-danger: #1a1010;
    --background-panel: #14161a;
    --background-panel-subtle: transparent;
    --shadow: #000000;
    --light-border: #2c3038;
    --border-color: #2c3038;
    --text: #d5d7dc;
    --text-strong: #f3f4f6;
    --text-soft: #8e929c;
    --popup-front: #f3f4f6;
    --emphasis: #8ea0b5;
    --emphasis-soft: #5d6d7e;
    --emphasis-text: #f3f4f6;
    --button-text: #e6e7eb;
    --button-background: #2a2e36;
    --button-foreground-disabled: #6d717a;
    --button-background-disabled: #1c1e24;
    --button-border: #2c3038;
    --range-track-color: #101114;
    --range-thumb-color: #8ea0b5;
    --noise-image: none;
}
"""

NECGRO_USER_CSS = """/* NECGRO user CSS. Tighter layout, no noise texture, less chrome.
   Registered as StyleSheetFiles and as the last NECGRO theme sheet. */
:root {
    --noise-image: none;
    --panel-gap: 4px;
    --button-shadow: none;
}

.nav-tabs,
#currentimagecontent,
#inputsidebartab_content,
#rightsidebarcontent,
#t2i_bottom_bar,
.browser-folder-tree-container,
.browser-fullcontent-container,
.noise-image-bg {
    background-image: none;
}

.nav-tabs {
    font-size: 14px;
    margin-bottom: 0;
}

.nav-tabs .nav-item a {
    padding: 2px 10px;
}

.t2i-area-quicktools {
    margin-top: 0;
    padding: 2px 8px !important;
    border-radius: 0 !important;
    box-shadow: none !important;
}

#main_inputs_area_wrapper {
    padding: 8px 10px;
}

.input-group .input-group-content {
    --spacing: 12px;
    margin-top: 4px;
}

.input-group.input-group-open .input-group-content {
    border-radius: 0 !important;
    background-color: transparent;
}

.alt_prompt_textbox {
    border-radius: 0;
    box-shadow: none;
    border: 1px solid var(--border-color);
}

.current_image_small {
    gap: 8px;
    padding-top: 8px !important;
    padding-bottom: 8px;
}

.current_image_batch_core {
    padding: 8px;
}

.browser_container .browser-folder-tree-container {
    padding: 8px;
}

.image-block,
.model-block,
.browser-header-bar input,
.section_wrapper,
.theme_preview .form-check {
    border-radius: 0;
}

.version-display {
    font-size: 12px;
    opacity: 0.55;
    background: transparent;
    padding: 0;
}

.install_q_head {
    font-size: 1.1rem;
}
"""

NECGRO_NAME_JS = """(function () {
    const swap = (value) => value.split('SwarmUI').join('NECGRO');
    const fixText = (root) => {
        if (!root) {
            return;
        }
        const walker = document.createTreeWalker(root, NodeFilter.SHOW_TEXT);
        let node;
        while ((node = walker.nextNode())) {
            if (node.nodeValue && node.nodeValue.indexOf('SwarmUI') !== -1) {
                node.nodeValue = swap(node.nodeValue);
            }
        }
    };
    const run = () => {
        if (document.title.indexOf('SwarmUI') !== -1) {
            document.title = swap(document.title);
        }
        fixText(document.getElementById('version_display'));
        fixText(document.getElementById('welcome_message'));
        fixText(document.querySelector('h1'));
    };
    const watch = (el) => {
        if (!el) {
            return;
        }
        new MutationObserver(run).observe(el, { childList: true, subtree: true, characterData: true });
    };
    run();
    watch(document.getElementById('current_image'));
    watch(document.getElementById('version_display'));
})();
"""

NECGRO_CS = """using SwarmUI.Core;

// Namespace must not contain "SwarmUI" (reserved for built-ins).
namespace NECGROSkin;

public class NECGRO : Extension
{
    public override void OnPreInit()
    {
        Version = "1";
        ExtensionAuthor = "NECGRO";
        Description = "Dark minimal UI skin.";
        License = "MIT";
        StyleSheetFiles.Add("Assets/necgro-user.css");
        ScriptFiles.Add("Assets/necgro-name.js");
        OtherAssets.Add("Assets/necgro.css");
    }

    public override void OnInit()
    {
        Program.Web.RegisterTheme(new WebServer.ThemeData(
            "necgro",
            "NECGRO",
            new string[]
            {
                "/css/themes/modern.css",
                "/ExtensionFile/NECGRO/Assets/necgro.css",
                "/ExtensionFile/NECGRO/Assets/necgro-user.css"
            },
            true));
    }
}
"""

NECGRO_CSPROJ = """<Project Sdk="Microsoft.NET.Sdk.Web">
    <PropertyGroup>
        <AssemblyName>NECGRO</AssemblyName>
    </PropertyGroup>
    <Import Project="../../SwarmUI.extension.props" />
</Project>
"""

_PATCHES = {
    "src/Pages/Shared/_Layout.cshtml": [
        (
            '<title id="page_title">@ViewData["Title"] - SwarmUI - @Program.ServerSettings.UserAuthorization.InstanceTitle</title>',
            '<title id="page_title">@ViewData["Title"] - NECGRO</title>',
            " - NECGRO</title>",
        ),
        (
            'string themeId = "modern_dark";',
            'string themeId = Program.Web.RegisteredThemes.ContainsKey("necgro") ? "necgro" : "modern_dark";',
            'ContainsKey("necgro")',
        ),
        (
            '<div id="version_display" class="version-display">SwarmUI v<span>',
            '<div id="version_display" class="version-display">NECGRO v<span>',
            "NECGRO v<span>",
        ),
    ],
    "src/Pages/Install.cshtml": [
        (
            'ViewData["Title"] = "SwarmUI Installer";',
            'ViewData["Title"] = "Install";',
            'ViewData["Title"] = "Install";',
        ),
        (
            "<h1>SwarmUI Installer</h1>",
            "<h1>NECGRO</h1>",
            "<h1>NECGRO</h1>",
        ),
        (
            'themeId == "modern_dark"',
            'themeId == "necgro"',
            'themeId == "necgro"',
        ),
    ],
    "src/wwwroot/js/installer.js": [
        (
            """        if (theme.startsWith('modern')) {
            css_paths.unshift('/css/themes/modern.css');
        }
""",
            """        if (theme.startsWith('modern')) {
            css_paths.unshift('/css/themes/modern.css');
        }
        if (theme === 'necgro') {
            css_paths = ['/css/themes/modern.css', '/ExtensionFile/NECGRO/Assets/necgro.css', '/ExtensionFile/NECGRO/Assets/necgro-user.css'];
            isDark = true;
        }
""",
            "theme === 'necgro'",
        ),
    ],
    "src/Pages/Login.cshtml": [
        (
            "Login to</span> SwarmUI:",
            "Login to</span> NECGRO:",
            "Login to</span> NECGRO:",
        ),
    ],
    "src/Pages/Register.cshtml": [
        (
            "Register an account for</span> SwarmUI:",
            "Register an account for</span> NECGRO:",
            "Register an account for</span> NECGRO:",
        ),
    ],
    "src/Pages/GoogleOAuthVerify.cshtml": [
        (
            "Register an account for</span> SwarmUI:",
            "Register an account for</span> NECGRO:",
            "Register an account for</span> NECGRO:",
        ),
    ],
}

_REQUIRED = {
    "src/Pages/Shared/_Layout.cshtml",
    "src/Pages/Install.cshtml",
    "src/wwwroot/js/installer.js",
}


def _looks_like_swarm(path):
    return os.path.isdir(os.path.join(path, "src", "wwwroot")) and os.path.isfile(
        os.path.join(path, "src", "Pages", "Shared", "_Layout.cshtml")
    )


def _resolve_swarm_root(swarm_root=None):
    candidates = []
    if swarm_root:
        candidates.append(swarm_root)
    candidates.append(os.getcwd())
    swarm_path = os.environ.get("SWARMPATH")
    if swarm_path:
        candidates.append(os.path.join(swarm_path, "SwarmUI"))
    candidates.extend(
        [
            "/content/drive/MyDrive/SwarmUI",
            "/content/SwarmUI",
            "/content/StableSwarmUI",
            "/content/drive/MyDrive/StableSwarmUI",
        ]
    )
    seen = set()
    for candidate in candidates:
        if not candidate or candidate in seen:
            continue
        seen.add(candidate)
        if _looks_like_swarm(candidate):
            return candidate
    raise FileNotFoundError("SwarmUI install not found. Clone it before applying the NECGRO skin.")


def _write(path, text):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def _patch_file(path, pairs):
    raw = path.read_bytes()
    text = raw.decode("utf-8")
    original = text
    for old, new, marker in pairs:
        if marker in text:
            continue
        if old not in text:
            print(f"NECGRO: skipped {path.name}; expected text was not found.")
            continue
        text = text.replace(old, new, 1)
    if text != original:
        path.write_bytes(text.encode("utf-8"))


def update_settings_text(text):
    newline = "\r\n" if "\r\n" in text else "\n"
    lines = text.splitlines()

    def indent_of(line):
        return len(line) - len(line.lstrip(" "))

    stack = []
    instance_idx = None
    theme_idx = None
    user_auth_idx = None
    default_user_idx = None
    for i, line in enumerate(lines):
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            continue
        ind = indent_of(line)
        while stack and stack[-1][0] >= ind:
            stack.pop()
        if ":" not in stripped:
            continue
        name, _, value = stripped.partition(":")
        name = name.strip()
        path = tuple(item[1] for item in stack)
        if name == "InstanceTitle" and path == ("UserAuthorization",):
            instance_idx = i
        if name == "Theme" and path == ("DefaultUser",):
            theme_idx = i
        if name == "UserAuthorization" and path == () and value.strip() == "":
            user_auth_idx = i
        if name == "DefaultUser" and path == () and value.strip() == "":
            default_user_idx = i
        if value.strip() == "":
            stack.append((ind, name))

    def set_line(idx, key, value):
        lines[idx] = (" " * indent_of(lines[idx])) + f"{key}: {value}"

    if instance_idx is not None:
        set_line(instance_idx, "InstanceTitle", "NECGRO")
    elif user_auth_idx is not None:
        lines.insert(user_auth_idx + 1, (" " * (indent_of(lines[user_auth_idx]) + 4)) + "InstanceTitle: NECGRO")
        if default_user_idx is not None and default_user_idx > user_auth_idx:
            default_user_idx += 1
        if theme_idx is not None and theme_idx > user_auth_idx:
            theme_idx += 1
    else:
        if lines and lines[-1].strip():
            lines.append("")
        lines.append("UserAuthorization:")
        lines.append("    InstanceTitle: NECGRO")

    if theme_idx is not None:
        set_line(theme_idx, "Theme", "necgro")
    elif default_user_idx is not None:
        lines.insert(default_user_idx + 1, (" " * (indent_of(lines[default_user_idx]) + 4)) + "Theme: necgro")
    else:
        if lines and lines[-1].strip():
            lines.append("")
        lines.append("DefaultUser:")
        lines.append("    Theme: necgro")

    body = newline.join(lines)
    if text.endswith(("\n", "\r\n")) or text == "":
        if not body.endswith("\n"):
            body += newline
    return body


def apply_necgro_skin(swarm_root=None):
    """Write the NECGRO extension, point settings at it, and rename visible SwarmUI chrome."""
    root = Path(_resolve_swarm_root(swarm_root))
    extension = root / "src" / "Extensions" / "NECGRO"
    _write(extension / "NECGRO.cs", NECGRO_CS)
    _write(extension / "NECGRO.csproj", NECGRO_CSPROJ)
    _write(extension / "Assets" / "necgro.css", NECGRO_THEME_CSS)
    _write(extension / "Assets" / "necgro-user.css", NECGRO_USER_CSS)
    _write(extension / "Assets" / "necgro-name.js", NECGRO_NAME_JS)

    for rel, pairs in _PATCHES.items():
        path = root / rel
        if not path.is_file():
            message = f"NECGRO: missing {rel}."
            if rel in _REQUIRED:
                raise FileNotFoundError(message)
            print(message)
            continue
        _patch_file(path, pairs)

    settings = root / "Data" / "Settings.fds"
    current = settings.read_text(encoding="utf-8") if settings.is_file() else ""
    _write(settings, update_settings_text(current))
    print(f"NECGRO skin applied in {root}")
    return str(root)

import os
import subprocess

# Construct the full path to SwarmUI
swarmui_full_path = os.path.join(os.environ['SWARMPATH'], 'SwarmUI')

# Change directory to SwarmUI installation path reliably
try:
    os.chdir(swarmui_full_path)
    print(f"Changed directory to: {os.getcwd()}")
except FileNotFoundError:
    print(f"Error: SwarmUI directory not found at {swarmui_full_path}. Please ensure it is cloned correctly.")
    # Exit or raise error if directory change fails
    raise

# (Colab stupid-proofing: aggressive git ultraforce pull)
# These commands require being in a valid git repository
print("Updating SwarmUI repository...")
!git fetch --all
!git clean -fdx # Remove untracked files and directories forcefully
!git reset --hard origin/main # Assuming 'main' is the default branch for mcmonkeyprojects/SwarmUI
!git pull

# (Colab stupid-proofing: drive wonks the perms so this needs an aggressive rebuild)
!rm -rf ./src/bin/live_release

# NECGRO skin. git clean -fdx deletes the ignored extension, so reapply after the pull.
apply_necgro_skin()

# Ensure dotnet is in PATH for the launch script and execute it
print("Launching NECGRO...")
!export DOTNET_ROOT=/root/.dotnet && export PATH=$DOTNET_ROOT:$PATH && bash ./launch-linux.sh --launch_mode none --cloudflared-path cloudflared

In [ ]:
import os
import shutil

# Dynamically search for the wildcards directory in possible paths
possible_paths = [
    '/content/StableSwarmUI/src/wwwroot/wildcards',
    '/content/StableSwarmUI/src/Assets/wildcards',
    '/content/drive/MyDrive/SwarmUI/src/wwwroot/wildcards',
    '/content/drive/MyDrive/StableSwarmUI/src/wwwroot/wildcards'
]

swarm_wildcards_path = None
for path in possible_paths:
    if os.path.exists(path):
        swarm_wildcards_path = path
        break

if not swarm_wildcards_path:
    # Deeper search fallback
    for root, dirs, files in os.walk('/content'):
        if 'wildcards' in dirs and 'StableSwarmUI' in root:
            swarm_wildcards_path = os.path.join(root, 'wildcards')
            break

print(f"Target wildcards path: {swarm_wildcards_path}")

if swarm_wildcards_path and os.path.exists(swarm_wildcards_path):
    sfw_path = os.path.join(swarm_wildcards_path, 'sfw')
    if os.path.exists(sfw_path):
        print("Found 'sfw' folder. Merging its contents into parent 'wildcards' directory to eliminate warnings...")
        for item in os.listdir(sfw_path):
            source = os.path.join(sfw_path, item)
            destination = os.path.join(swarm_wildcards_path, item)
            if os.path.isfile(source):
                try:
                    shutil.copy2(source, destination)
                    print(f"Copied: {item} -> root wildcards folder")
                except Exception as e:
                    print(f"Failed to copy {item}: {e}")
    else:
        # Let's create the folder and fetch/write mock wildcards or look elsewhere if they aren't generated yet
        print("Could not find 'sfw' subfolder inside wildcards. It might be created on initial run.")
else:
    print("Wildcards folder not found. SwarmUI may need to be initialized or launched first so it downloads default wildcards.")

In [ ]:
import os

# Dynamically locate the active SwarmUI/StableSwarmUI directory
swarm_paths = [
    '/content/drive/MyDrive/SwarmUI',
    '/content/StableSwarmUI',
    '/content/SwarmUI',
    '/content/drive/MyDrive/StableSwarmUI'
]

base_path = None
for path in swarm_paths:
    if os.path.exists(path):
        base_path = path
        break

if base_path:
    wildcards_path = os.path.join(base_path, 'src/wwwroot/wildcards')
    os.makedirs(wildcards_path, exist_ok=True)
    os.makedirs(os.path.join(wildcards_path, 'sfw'), exist_ok=True)
    print(f"Target wildcards folder identified: {wildcards_path}")

    # Structure wildcards as dictionary { filepath: list_of_lines }
    wildcard_templates = {
        'creature_adjectives.txt': [
            "mythical", "robotic", "bioluminescent", "ancient", "spectral",
            "etherial", "armored", "draconic", "crystalline", "shadowy"
        ],
        'creature_types.txt': [
            "beast", "golem", "dragon", "serpent", "humanoid",
            "griffin", "phoenix", "chimera", "construct", "spirit"
        ],
        'creature_anatomy.txt': [
            "glowing eyes and majestic wings",
            "sharp metallic claws and horns",
            "translucent crystalline scales",
            "a crown of blue fire and smoky tail",
            "intricate runic engravings across its chassis"
        ],
        'creature_actions.txt': [
            "roaring into the sky",
            "emerging from a dense fog",
            "channeling ancient magic energy",
            "resting atop a mountain peak",
            "stalking through a glowing forest"
        ],
        'sfw/element.txt': [
            "fire", "water", "earth", "air", "lightning",
            "ice", "light", "shadow", "cosmic energy", "nature"
        ],
        'sfw/landscape.txt': [
            "volcanic peaks", "crystal caverns", "floating sky islands",
            "cyberpunk ruins", "ancient redwood forest", "bioluminescent ocean"
        ],
        'sfw/atmosphere.txt': [
            "mysterious", "epic", "serene", "dark fantasy",
            "cyberpunk", "dreamlike", "cinematic", "ethereal"
        ],
        'sfw/lighting.txt': [
            "volumetric rays", "dramatic chiaroscuro", "neon glow",
            "golden hour", "bioluminescent ambient light", "moonlit"
        ],
        'sfw/particle_effect.txt': [
            "floating embers", "glowing runes", "swirling stardust",
            "electrical sparks", "falling autumn leaves", "misty vapor"
        ],
        'sfw/genre.txt': [
            "high fantasy", "science fiction", "steampunk",
            "dark fantasy", "cosmic horror", "mythological"
        ],
        'sfw/art_style.txt': [
            "detailed digital illustration", "concept art paint over",
            "octane render 3D", "unreal engine 5 scene", "fine oil painting"
        ],
        'sfw/camera.txt': [
            "extreme close-up", "wide-angle cinematic shot",
            "isometric view", "low-angle dramatic shot", "macro photography"
        ]
    }

    # Write out each wildcard template
    for rel_path, content in wildcard_templates.items():
        full_file_path = os.path.join(wildcards_path, rel_path)
        os.makedirs(os.path.dirname(full_file_path), exist_ok=True)
        with open(full_file_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(content) + '\n')
        print(f"Successfully generated: {rel_path} ({len(content)} entries)")
else:
    print("Could not automatically locate your SwarmUI installation directory. Please double-check your pathing.")

In [ ]:
import os
import shutil

# Detect active SwarmUI base path
# Prioritize the SWARMPATH set in previous cells (which should be in Drive)
swarm_paths = [
    os.path.join(os.environ.get('SWARMPATH', '/content'), 'SwarmUI'),
    '/content/StableSwarmUI',
    '/content/drive/MyDrive/SwarmUI',
    '/content/SwarmUI',
    '/content/drive/MyDrive/StableSwarmUI'
]

base_path = None
for path in swarm_paths:
    # Check for the existence of a key subdirectory to confirm it's a SwarmUI installation
    if os.path.exists(os.path.join(path, 'src', 'wwwroot')):
        base_path = path
        break

if base_path:
    wildcards_path = os.path.join(base_path, 'src/wwwroot/wildcards')

    # Purge existing wildcard directory if it exists to perform a clean database rebuild
    if os.path.exists(wildcards_path):
        print(f"Purging old wildcard database at: {wildcards_path}...")
        shutil.rmtree(wildcards_path)

    os.makedirs(wildcards_path, exist_ok=True)
    os.makedirs(os.path.join(wildcards_path, 'sfw'), exist_ok=True)
    print(f"Created fresh target wildcards directory at: {wildcards_path}")

    # Define the renewed and comprehensive database
    wildcard_database = {
        'creature_adjectives.txt': [
            "mythical", "robotic", "bioluminescent", "ancient", "spectral",
            "ethereal", "armored", "draconic", "crystalline", "shadowy",
            "celestial", "corrupted", "frostbound", "magmatic", "clockwork"
        ],
        'creature_types.txt': [
            "beast", "golem", "dragon", "serpent", "humanoid",
            "griffin", "phoenix", "chimera", "construct", "spirit",
            "colossus", "leviathan", "wraith", "wyvern", "sentinel"
        ],
        'creature_anatomy.txt': [
            "glowing eyes and majestic wings",
            "sharp metallic claws and horns",
            "translucent crystalline scales",
            "a crown of blue fire and smoky tail",
            "intricate runic engravings across its chassis",
            "overgrown mossy armor plating",
            "crackling electrical nodes running down its spine"
        ],
        'creature_actions.txt': [
            "roaring into the sky",
            "emerging from a dense fog",
            "channeling ancient magic energy",
            "resting atop a mountain peak",
            "stalking through a glowing forest",
            "ascending into a storm cloud",
            "guarding an ancient golden chest"
        ],
        'sfw/element.txt': [
            "fire", "water", "earth", "air", "lightning",
            "ice", "light", "shadow", "cosmic energy", "nature",
            "magma", "plasma", "gravity", "time", "spectral essence"
        ],
        'sfw/landscape.txt': [
            "volcanic peaks", "crystal caverns", "floating sky islands",
            "cyberpunk ruins", "ancient redwood forest", "bioluminescent ocean",
            "steaming geothermal geysers", "windswept desert dunes", "overgrown stone temples"
        ],
        'sfw/atmosphere.txt': [
            "mysterious", "epic", "serene", "dark fantasy",
            "cyberpunk", "dreamlike", "cinematic", "ethereal",
            "majestic", "ominous", "nostalgic", "surreal"
        ],
        'sfw/lighting.txt': [
            "volumetric rays", "dramatic chiaroscuro", "neon glow",
            "golden hour", "bioluminescent ambient light", "moonlit",
            "sunshafts filtering through trees", "intense rim lighting"
        ],
        'sfw/particle_effect.txt': [
            "floating embers", "glowing runes", "swirling stardust",
            "electrical sparks", "falling autumn leaves", "misty vapor",
            "shattered glass shards", "floating cherry blossom petals"
        ],
        'sfw/genre.txt': [
            "high fantasy", "science fiction", "steampunk",
            "dark fantasy", "cosmic horror", "mythological",
            "dieselpunk", "cyber-noir", "solarpunk"
        ],
        'sfw/art_style.txt': [
            "detailed digital illustration", "concept art paint over",
            "octane render 3D", "unreal engine 5 scene", "fine oil painting",
            "watercolor and ink sketching", "retro anime style cel shading"
        ],
        'sfw/camera.txt': [
            "extreme close-up", "wide-angle cinematic shot",
            "isometric view", "low-angle dramatic shot", "macro photography",
            "gopro action capture", "satellite orbital view"
        ],
        'templates.txt': [
            "A <creature_adjectives> <creature_types> with <creature_anatomy>, <creature_actions> in <sfw/landscape>, <sfw/atmosphere> atmosphere, <sfw/lighting> lighting, <sfw/particle_effect> particle effects, <sfw/genre> style, <sfw/art_style>, <sfw/camera>",
            "Epic <sfw/art_style> of a <creature_adjectives> <creature_types>, elemental power of <sfw/element>, dramatic <sfw/lighting>, cinematic <sfw/camera>",
            "A majestic <creature_types> showing <creature_anatomy>, <creature_actions>, <sfw/particle_effect> details, set in a surreal <sfw/landscape>, <sfw/genre> genre",
            "Close-up <sfw/camera> of <creature_adjectives> <creature_types> infused with <sfw/element>, glowing runes, <sfw/art_style>"
        ]
    }

    # Generate new database files and synchronize them
    for rel_path, items in wildcard_database.items():
        full_file_path = os.path.join(wildcards_path, rel_path)
        os.makedirs(os.path.dirname(full_file_path), exist_ok=True)
        with open(full_file_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(items) + '\n')

        # If it's a file inside a subdirectory, copy a duplicate to the root directory
        # to support flat format queries/compatibility issues with some nodes
        if '/' in rel_path or '\\' in rel_path:
            flat_filename = os.path.basename(rel_path)
            flat_target = os.path.join(wildcards_path, flat_filename)
            shutil.copy2(full_file_path, flat_target)

    print("\nAll wildcard database files (including the templates wildcard) have been completely rebuilt and mirrored to the root directory.")
else:
    print("Could not automatically locate your SwarmUI directory. Please check that SwarmUI is installed correctly. Make sure SwarmUI is cloned into '/content/drive/MyDrive/SwarmUI'.")

In [ ]:
# Pre-install ComfyUI dependencies globally to resolve the 'ModuleNotFoundError: No module named filelock' loop
!pip install filelock Pillow torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121

# Also install the requirements directly from the ComfyUI directory if it exists
import os
comfy_reqs = '/content/StableSwarmUI/dlbackend/ComfyUI/requirements.txt'
if os.path.exists(comfy_reqs):
    print("Installing requirements from ComfyUI requirements.txt...")
    !pip install -r {comfy_reqs}
else:
    print("ComfyUI requirements.txt not found yet. It will be created when SwarmUI fully runs or downloads it.")

### SwarmUI Installation Path Selection
Please select your preferred installation path for SwarmUI. Choosing 'Google Drive (Persistent)' will mount your Google Drive and install SwarmUI there, allowing your data to persist across sessions. 'Temporary (/content/)' will install SwarmUI in the Colab instance's temporary storage, which will be lost when the session ends.

In [ ]:
# This cell was identified as cloning a conflicting SwarmUI version (Stability-AI/StableSwarmUI) to a temporary path, which caused `ModuleNotFoundError` for `filelock` and interfered with the intended `mcmonkeyprojects/SwarmUI` installation. Its content has been cleared to ensure consistent installation of the intended SwarmUI to Google Drive.

In [ ]:
import os

drive_checkpoints = '/content/drive/MyDrive/SwarmColab/Models/checkpoints'
drive_loras = '/content/drive/MyDrive/SwarmColab/Models/loras'

swarm_checkpoints = '/content/StableSwarmUI/Models/Stable-Diffusion'
swarm_loras = '/content/StableSwarmUI/Models/Lora'

print("=== Persistent Google Drive Model Files ===")
if os.path.exists(drive_checkpoints):
    checkpoints = os.listdir(drive_checkpoints)
    print(f"Checkpoints in Drive ({len(checkpoints)}): {checkpoints}")
else:
    print("Drive checkpoints folder does not exist.")

if os.path.exists(drive_loras):
    loras = os.listdir(drive_loras)
    print(f"Loras in Drive ({len(loras)}): {loras}")
else:
    print("Drive loras folder does not exist.")

print("\n=== Active SwarmUI Directory Symlinks ===")
for name, path in [("Checkpoints", swarm_checkpoints), ("Loras", swarm_loras)]:
    if os.path.islink(path):
        target = os.readlink(path)
        print(f"{name} symlink exists and points to: {target}")
        try:
            resolved_files = os.listdir(path)
            print(f"  Resolved files through symlink ({len(resolved_files)}): {resolved_files}")
        except Exception as e:
            print(f"  Error reading files through symlink: {e}")
    elif os.path.exists(path):
        print(f"{name} path exists but is a standard directory (not a symlink).")
    else:
        print(f"{name} path does not exist.")

# New Section

In [ ]:
#### PICK ONE: *OR* this node for install in google drive
from google.colab import drive
drive.mount('/content/drive')
SWARMPATH = '/content/drive/MyDrive'

# Colab breaks venvs, and doesn't save anything valid to drive, so just screw it do a global install of pip reqs if we used a comfy backend in the drive
!if [[ -f "$SWARMPATH/SwarmUI/dlbackend/ComfyUI/requirements.txt" ]]; then rm -rf $SWARMPATH/SwarmUI/dlbackend/ComfyUI/venv/; pip install -r $SWARMPATH/SwarmUI/dlbackend/ComfyUI/requirements.txt; fi

In [ ]:
import os
import shutil

# Install dotnet dependencies
!wget https://dot.net/v1/dotnet-install.sh -O dotnet-install.sh
!chmod +x dotnet-install.sh
!./dotnet-install.sh --channel 8.0

# Set DOTNET_ROOT and add to PATH to ensure dotnet is found for current session and sub-shells
os.environ['DOTNET_ROOT'] = '/root/.dotnet'
os.environ['PATH'] = os.environ['DOTNET_ROOT'] + ':' + os.environ['PATH']
!export DOTNET_ROOT=/root/.dotnet
!export PATH=$DOTNET_ROOT:$PATH

# Install Cloudflared
!wget -nc https://github.com/cloudflare/cloudflared/releases/download/2024.8.2/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# Use the chosen path from previous cells (SWARMPATH is expected to be '/content/drive/MyDrive')
os.environ['SWARMPATH'] = SWARMPATH
# Ensure we are in the base SWARMPATH before attempting SwarmUI operations
%cd $SWARMPATH

# Colab breaks venv, so, tell swarm to not make a venv
os.environ['SWARM_NO_VENV'] = 'true'

# Download SwarmUI (mcmonkeyprojects/SwarmUI)
url = "https://github.com/mcmonkeyprojects/SwarmUI"
swarmui_target_path = os.path.join(SWARMPATH, 'SwarmUI')

# Check if SwarmUI directory exists and is a valid git repository
if os.path.exists(swarmui_target_path):
    # If it exists, but isn't a valid git repo, remove and re-clone
    if not os.path.isdir(os.path.join(swarmui_target_path, '.git')):
        print(f"Existing SwarmUI at {swarmui_target_path} is not a valid git repository. Removing and re-cloning...")
        shutil.rmtree(swarmui_target_path)
        !git clone $url $swarmui_target_path
    else:
        print(f"SwarmUI already exists at {swarmui_target_path} and is a valid git repository, skipping clone.")
else:
    print(f"Cloning SwarmUI into {swarmui_target_path}...")
    !git clone $url $swarmui_target_path

# Verify the SwarmUI directory exists after cloning/checking
if not os.path.exists(swarmui_target_path):
    raise FileNotFoundError(f"SwarmUI directory not found at {swarmui_target_path} after cloning attempt.")

In [ ]:
import os
from pathlib import Path


# SwarmUI applies a theme by registering CSS paths (WebServer.RegisterTheme) and
# selecting one from the user Theme setting. The page layout falls back to
# modern_dark until that setting or the sui_theme_id cookie is set.
# Extra user CSS is an extension StyleSheetFiles entry, injected into the page
# header. Theme stylesheets are separate and load after that, so the layout
# sheet is also the last NECGRO theme file and wins over modern.css.
# The visible product name is hardcoded as "SwarmUI" in the layout title and
# version line. UserAuthorization.InstanceTitle is the instance name beside it.

NECGRO_THEME_CSS = """/* NECGRO theme colors. Dark and flat. */
:root {
    --background: #101114;
    --background-soft: #181a1f;
    --background-gray: #1c1e24;
    --background-gray-danger: #2a1818;
    --background-danger: #1a1010;
    --background-panel: #14161a;
    --background-panel-subtle: transparent;
    --shadow: #000000;
    --light-border: #2c3038;
    --border-color: #2c3038;
    --text: #d5d7dc;
    --text-strong: #f3f4f6;
    --text-soft: #8e929c;
    --popup-front: #f3f4f6;
    --emphasis: #8ea0b5;
    --emphasis-soft: #5d6d7e;
    --emphasis-text: #f3f4f6;
    --button-text: #e6e7eb;
    --button-background: #2a2e36;
    --button-foreground-disabled: #6d717a;
    --button-background-disabled: #1c1e24;
    --button-border: #2c3038;
    --range-track-color: #101114;
    --range-thumb-color: #8ea0b5;
    --noise-image: none;
}
"""

NECGRO_USER_CSS = """/* NECGRO user CSS. Tighter layout, no noise texture, less chrome.
   Registered as StyleSheetFiles and as the last NECGRO theme sheet. */
:root {
    --noise-image: none;
    --panel-gap: 4px;
    --button-shadow: none;
}

.nav-tabs,
#currentimagecontent,
#inputsidebartab_content,
#rightsidebarcontent,
#t2i_bottom_bar,
.browser-folder-tree-container,
.browser-fullcontent-container,
.noise-image-bg {
    background-image: none;
}

.nav-tabs {
    font-size: 14px;
    margin-bottom: 0;
}

.nav-tabs .nav-item a {
    padding: 2px 10px;
}

.t2i-area-quicktools {
    margin-top: 0;
    padding: 2px 8px !important;
    border-radius: 0 !important;
    box-shadow: none !important;
}

#main_inputs_area_wrapper {
    padding: 8px 10px;
}

.input-group .input-group-content {
    --spacing: 12px;
    margin-top: 4px;
}

.input-group.input-group-open .input-group-content {
    border-radius: 0 !important;
    background-color: transparent;
}

.alt_prompt_textbox {
    border-radius: 0;
    box-shadow: none;
    border: 1px solid var(--border-color);
}

.current_image_small {
    gap: 8px;
    padding-top: 8px !important;
    padding-bottom: 8px;
}

.current_image_batch_core {
    padding: 8px;
}

.browser_container .browser-folder-tree-container {
    padding: 8px;
}

.image-block,
.model-block,
.browser-header-bar input,
.section_wrapper,
.theme_preview .form-check {
    border-radius: 0;
}

.version-display {
    font-size: 12px;
    opacity: 0.55;
    background: transparent;
    padding: 0;
}

.install_q_head {
    font-size: 1.1rem;
}
"""

NECGRO_NAME_JS = """(function () {
    const swap = (value) => value.split('SwarmUI').join('NECGRO');
    const fixText = (root) => {
        if (!root) {
            return;
        }
        const walker = document.createTreeWalker(root, NodeFilter.SHOW_TEXT);
        let node;
        while ((node = walker.nextNode())) {
            if (node.nodeValue && node.nodeValue.indexOf('SwarmUI') !== -1) {
                node.nodeValue = swap(node.nodeValue);
            }
        }
    };
    const run = () => {
        if (document.title.indexOf('SwarmUI') !== -1) {
            document.title = swap(document.title);
        }
        fixText(document.getElementById('version_display'));
        fixText(document.getElementById('welcome_message'));
        fixText(document.querySelector('h1'));
    };
    const watch = (el) => {
        if (!el) {
            return;
        }
        new MutationObserver(run).observe(el, { childList: true, subtree: true, characterData: true });
    };
    run();
    watch(document.getElementById('current_image'));
    watch(document.getElementById('version_display'));
})();
"""

NECGRO_CS = """using SwarmUI.Core;

// Namespace must not contain "SwarmUI" (reserved for built-ins).
namespace NECGROSkin;

public class NECGRO : Extension
{
    public override void OnPreInit()
    {
        Version = "1";
        ExtensionAuthor = "NECGRO";
        Description = "Dark minimal UI skin.";
        License = "MIT";
        StyleSheetFiles.Add("Assets/necgro-user.css");
        ScriptFiles.Add("Assets/necgro-name.js");
        OtherAssets.Add("Assets/necgro.css");
    }

    public override void OnInit()
    {
        Program.Web.RegisterTheme(new WebServer.ThemeData(
            "necgro",
            "NECGRO",
            new string[]
            {
                "/css/themes/modern.css",
                "/ExtensionFile/NECGRO/Assets/necgro.css",
                "/ExtensionFile/NECGRO/Assets/necgro-user.css"
            },
            true));
    }
}
"""

NECGRO_CSPROJ = """<Project Sdk="Microsoft.NET.Sdk.Web">
    <PropertyGroup>
        <AssemblyName>NECGRO</AssemblyName>
    </PropertyGroup>
    <Import Project="../../SwarmUI.extension.props" />
</Project>
"""

_PATCHES = {
    "src/Pages/Shared/_Layout.cshtml": [
        (
            '<title id="page_title">@ViewData["Title"] - SwarmUI - @Program.ServerSettings.UserAuthorization.InstanceTitle</title>',
            '<title id="page_title">@ViewData["Title"] - NECGRO</title>',
            " - NECGRO</title>",
        ),
        (
            'string themeId = "modern_dark";',
            'string themeId = Program.Web.RegisteredThemes.ContainsKey("necgro") ? "necgro" : "modern_dark";',
            'ContainsKey("necgro")',
        ),
        (
            '<div id="version_display" class="version-display">SwarmUI v<span>',
            '<div id="version_display" class="version-display">NECGRO v<span>',
            "NECGRO v<span>",
        ),
    ],
    "src/Pages/Install.cshtml": [
        (
            'ViewData["Title"] = "SwarmUI Installer";',
            'ViewData["Title"] = "Install";',
            'ViewData["Title"] = "Install";',
        ),
        (
            "<h1>SwarmUI Installer</h1>",
            "<h1>NECGRO</h1>",
            "<h1>NECGRO</h1>",
        ),
        (
            'themeId == "modern_dark"',
            'themeId == "necgro"',
            'themeId == "necgro"',
        ),
    ],
    "src/wwwroot/js/installer.js": [
        (
            """        if (theme.startsWith('modern')) {
            css_paths.unshift('/css/themes/modern.css');
        }
""",
            """        if (theme.startsWith('modern')) {
            css_paths.unshift('/css/themes/modern.css');
        }
        if (theme === 'necgro') {
            css_paths = ['/css/themes/modern.css', '/ExtensionFile/NECGRO/Assets/necgro.css', '/ExtensionFile/NECGRO/Assets/necgro-user.css'];
            isDark = true;
        }
""",
            "theme === 'necgro'",
        ),
    ],
    "src/Pages/Login.cshtml": [
        (
            "Login to</span> SwarmUI:",
            "Login to</span> NECGRO:",
            "Login to</span> NECGRO:",
        ),
    ],
    "src/Pages/Register.cshtml": [
        (
            "Register an account for</span> SwarmUI:",
            "Register an account for</span> NECGRO:",
            "Register an account for</span> NECGRO:",
        ),
    ],
    "src/Pages/GoogleOAuthVerify.cshtml": [
        (
            "Register an account for</span> SwarmUI:",
            "Register an account for</span> NECGRO:",
            "Register an account for</span> NECGRO:",
        ),
    ],
}

_REQUIRED = {
    "src/Pages/Shared/_Layout.cshtml",
    "src/Pages/Install.cshtml",
    "src/wwwroot/js/installer.js",
}


def _looks_like_swarm(path):
    return os.path.isdir(os.path.join(path, "src", "wwwroot")) and os.path.isfile(
        os.path.join(path, "src", "Pages", "Shared", "_Layout.cshtml")
    )


def _resolve_swarm_root(swarm_root=None):
    candidates = []
    if swarm_root:
        candidates.append(swarm_root)
    candidates.append(os.getcwd())
    swarm_path = os.environ.get("SWARMPATH")
    if swarm_path:
        candidates.append(os.path.join(swarm_path, "SwarmUI"))
    candidates.extend(
        [
            "/content/drive/MyDrive/SwarmUI",
            "/content/SwarmUI",
            "/content/StableSwarmUI",
            "/content/drive/MyDrive/StableSwarmUI",
        ]
    )
    seen = set()
    for candidate in candidates:
        if not candidate or candidate in seen:
            continue
        seen.add(candidate)
        if _looks_like_swarm(candidate):
            return candidate
    raise FileNotFoundError("SwarmUI install not found. Clone it before applying the NECGRO skin.")


def _write(path, text):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def _patch_file(path, pairs):
    raw = path.read_bytes()
    text = raw.decode("utf-8")
    original = text
    for old, new, marker in pairs:
        if marker in text:
            continue
        if old not in text:
            print(f"NECGRO: skipped {path.name}; expected text was not found.")
            continue
        text = text.replace(old, new, 1)
    if text != original:
        path.write_bytes(text.encode("utf-8"))


def update_settings_text(text):
    newline = "\r\n" if "\r\n" in text else "\n"
    lines = text.splitlines()

    def indent_of(line):
        return len(line) - len(line.lstrip(" "))

    stack = []
    instance_idx = None
    theme_idx = None
    user_auth_idx = None
    default_user_idx = None
    for i, line in enumerate(lines):
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            continue
        ind = indent_of(line)
        while stack and stack[-1][0] >= ind:
            stack.pop()
        if ":" not in stripped:
            continue
        name, _, value = stripped.partition(":")
        name = name.strip()
        path = tuple(item[1] for item in stack)
        if name == "InstanceTitle" and path == ("UserAuthorization",):
            instance_idx = i
        if name == "Theme" and path == ("DefaultUser",):
            theme_idx = i
        if name == "UserAuthorization" and path == () and value.strip() == "":
            user_auth_idx = i
        if name == "DefaultUser" and path == () and value.strip() == "":
            default_user_idx = i
        if value.strip() == "":
            stack.append((ind, name))

    def set_line(idx, key, value):
        lines[idx] = (" " * indent_of(lines[idx])) + f"{key}: {value}"

    if instance_idx is not None:
        set_line(instance_idx, "InstanceTitle", "NECGRO")
    elif user_auth_idx is not None:
        lines.insert(user_auth_idx + 1, (" " * (indent_of(lines[user_auth_idx]) + 4)) + "InstanceTitle: NECGRO")
        if default_user_idx is not None and default_user_idx > user_auth_idx:
            default_user_idx += 1
        if theme_idx is not None and theme_idx > user_auth_idx:
            theme_idx += 1
    else:
        if lines and lines[-1].strip():
            lines.append("")
        lines.append("UserAuthorization:")
        lines.append("    InstanceTitle: NECGRO")

    if theme_idx is not None:
        set_line(theme_idx, "Theme", "necgro")
    elif default_user_idx is not None:
        lines.insert(default_user_idx + 1, (" " * (indent_of(lines[default_user_idx]) + 4)) + "Theme: necgro")
    else:
        if lines and lines[-1].strip():
            lines.append("")
        lines.append("DefaultUser:")
        lines.append("    Theme: necgro")

    body = newline.join(lines)
    if text.endswith(("\n", "\r\n")) or text == "":
        if not body.endswith("\n"):
            body += newline
    return body


def apply_necgro_skin(swarm_root=None):
    """Write the NECGRO extension, point settings at it, and rename visible SwarmUI chrome."""
    root = Path(_resolve_swarm_root(swarm_root))
    extension = root / "src" / "Extensions" / "NECGRO"
    _write(extension / "NECGRO.cs", NECGRO_CS)
    _write(extension / "NECGRO.csproj", NECGRO_CSPROJ)
    _write(extension / "Assets" / "necgro.css", NECGRO_THEME_CSS)
    _write(extension / "Assets" / "necgro-user.css", NECGRO_USER_CSS)
    _write(extension / "Assets" / "necgro-name.js", NECGRO_NAME_JS)

    for rel, pairs in _PATCHES.items():
        path = root / rel
        if not path.is_file():
            message = f"NECGRO: missing {rel}."
            if rel in _REQUIRED:
                raise FileNotFoundError(message)
            print(message)
            continue
        _patch_file(path, pairs)

    settings = root / "Data" / "Settings.fds"
    current = settings.read_text(encoding="utf-8") if settings.is_file() else ""
    _write(settings, update_settings_text(current))
    print(f"NECGRO skin applied in {root}")
    return str(root)

# Alright, launch it! Watch the output for a Cloudflare URL
%cd $SWARMPATH/SwarmUI/

# (Colab stupid-proofing: aggressive git ultraforce pull)
!git fetch --all
!git reset --hard origin/main # Assuming 'main' is the default branch for mcmonkeyprojects/SwarmUI
!git pull

# (Colab stupid-proofing: drive wonks the perms so this needs an aggressive rebuild)
!rm -rf ./src/bin/live_release

# NECGRO skin. git reset restores upstream layout files, so reapply after the pull.
apply_necgro_skin()

!bash ./launch-linux.sh --launch_mode none --cloudflared-path cloudflared

In [ ]:
print('Checking for running SwarmUI process...')
!ps aux | grep "SwarmUI"

In [ ]:
import os
from pathlib import Path


# SwarmUI applies a theme by registering CSS paths (WebServer.RegisterTheme) and
# selecting one from the user Theme setting. The page layout falls back to
# modern_dark until that setting or the sui_theme_id cookie is set.
# Extra user CSS is an extension StyleSheetFiles entry, injected into the page
# header. Theme stylesheets are separate and load after that, so the layout
# sheet is also the last NECGRO theme file and wins over modern.css.
# The visible product name is hardcoded as "SwarmUI" in the layout title and
# version line. UserAuthorization.InstanceTitle is the instance name beside it.

NECGRO_THEME_CSS = """/* NECGRO theme colors. Dark and flat. */
:root {
    --background: #101114;
    --background-soft: #181a1f;
    --background-gray: #1c1e24;
    --background-gray-danger: #2a1818;
    --background-danger: #1a1010;
    --background-panel: #14161a;
    --background-panel-subtle: transparent;
    --shadow: #000000;
    --light-border: #2c3038;
    --border-color: #2c3038;
    --text: #d5d7dc;
    --text-strong: #f3f4f6;
    --text-soft: #8e929c;
    --popup-front: #f3f4f6;
    --emphasis: #8ea0b5;
    --emphasis-soft: #5d6d7e;
    --emphasis-text: #f3f4f6;
    --button-text: #e6e7eb;
    --button-background: #2a2e36;
    --button-foreground-disabled: #6d717a;
    --button-background-disabled: #1c1e24;
    --button-border: #2c3038;
    --range-track-color: #101114;
    --range-thumb-color: #8ea0b5;
    --noise-image: none;
}
"""

NECGRO_USER_CSS = """/* NECGRO user CSS. Tighter layout, no noise texture, less chrome.
   Registered as StyleSheetFiles and as the last NECGRO theme sheet. */
:root {
    --noise-image: none;
    --panel-gap: 4px;
    --button-shadow: none;
}

.nav-tabs,
#currentimagecontent,
#inputsidebartab_content,
#rightsidebarcontent,
#t2i_bottom_bar,
.browser-folder-tree-container,
.browser-fullcontent-container,
.noise-image-bg {
    background-image: none;
}

.nav-tabs {
    font-size: 14px;
    margin-bottom: 0;
}

.nav-tabs .nav-item a {
    padding: 2px 10px;
}

.t2i-area-quicktools {
    margin-top: 0;
    padding: 2px 8px !important;
    border-radius: 0 !important;
    box-shadow: none !important;
}

#main_inputs_area_wrapper {
    padding: 8px 10px;
}

.input-group .input-group-content {
    --spacing: 12px;
    margin-top: 4px;
}

.input-group.input-group-open .input-group-content {
    border-radius: 0 !important;
    background-color: transparent;
}

.alt_prompt_textbox {
    border-radius: 0;
    box-shadow: none;
    border: 1px solid var(--border-color);
}

.current_image_small {
    gap: 8px;
    padding-top: 8px !important;
    padding-bottom: 8px;
}

.current_image_batch_core {
    padding: 8px;
}

.browser_container .browser-folder-tree-container {
    padding: 8px;
}

.image-block,
.model-block,
.browser-header-bar input,
.section_wrapper,
.theme_preview .form-check {
    border-radius: 0;
}

.version-display {
    font-size: 12px;
    opacity: 0.55;
    background: transparent;
    padding: 0;
}

.install_q_head {
    font-size: 1.1rem;
}
"""

NECGRO_NAME_JS = """(function () {
    const swap = (value) => value.split('SwarmUI').join('NECGRO');
    const fixText = (root) => {
        if (!root) {
            return;
        }
        const walker = document.createTreeWalker(root, NodeFilter.SHOW_TEXT);
        let node;
        while ((node = walker.nextNode())) {
            if (node.nodeValue && node.nodeValue.indexOf('SwarmUI') !== -1) {
                node.nodeValue = swap(node.nodeValue);
            }
        }
    };
    const run = () => {
        if (document.title.indexOf('SwarmUI') !== -1) {
            document.title = swap(document.title);
        }
        fixText(document.getElementById('version_display'));
        fixText(document.getElementById('welcome_message'));
        fixText(document.querySelector('h1'));
    };
    const watch = (el) => {
        if (!el) {
            return;
        }
        new MutationObserver(run).observe(el, { childList: true, subtree: true, characterData: true });
    };
    run();
    watch(document.getElementById('current_image'));
    watch(document.getElementById('version_display'));
})();
"""

NECGRO_CS = """using SwarmUI.Core;

// Namespace must not contain "SwarmUI" (reserved for built-ins).
namespace NECGROSkin;

public class NECGRO : Extension
{
    public override void OnPreInit()
    {
        Version = "1";
        ExtensionAuthor = "NECGRO";
        Description = "Dark minimal UI skin.";
        License = "MIT";
        StyleSheetFiles.Add("Assets/necgro-user.css");
        ScriptFiles.Add("Assets/necgro-name.js");
        OtherAssets.Add("Assets/necgro.css");
    }

    public override void OnInit()
    {
        Program.Web.RegisterTheme(new WebServer.ThemeData(
            "necgro",
            "NECGRO",
            new string[]
            {
                "/css/themes/modern.css",
                "/ExtensionFile/NECGRO/Assets/necgro.css",
                "/ExtensionFile/NECGRO/Assets/necgro-user.css"
            },
            true));
    }
}
"""

NECGRO_CSPROJ = """<Project Sdk="Microsoft.NET.Sdk.Web">
    <PropertyGroup>
        <AssemblyName>NECGRO</AssemblyName>
    </PropertyGroup>
    <Import Project="../../SwarmUI.extension.props" />
</Project>
"""

_PATCHES = {
    "src/Pages/Shared/_Layout.cshtml": [
        (
            '<title id="page_title">@ViewData["Title"] - SwarmUI - @Program.ServerSettings.UserAuthorization.InstanceTitle</title>',
            '<title id="page_title">@ViewData["Title"] - NECGRO</title>',
            " - NECGRO</title>",
        ),
        (
            'string themeId = "modern_dark";',
            'string themeId = Program.Web.RegisteredThemes.ContainsKey("necgro") ? "necgro" : "modern_dark";',
            'ContainsKey("necgro")',
        ),
        (
            '<div id="version_display" class="version-display">SwarmUI v<span>',
            '<div id="version_display" class="version-display">NECGRO v<span>',
            "NECGRO v<span>",
        ),
    ],
    "src/Pages/Install.cshtml": [
        (
            'ViewData["Title"] = "SwarmUI Installer";',
            'ViewData["Title"] = "Install";',
            'ViewData["Title"] = "Install";',
        ),
        (
            "<h1>SwarmUI Installer</h1>",
            "<h1>NECGRO</h1>",
            "<h1>NECGRO</h1>",
        ),
        (
            'themeId == "modern_dark"',
            'themeId == "necgro"',
            'themeId == "necgro"',
        ),
    ],
    "src/wwwroot/js/installer.js": [
        (
            """        if (theme.startsWith('modern')) {
            css_paths.unshift('/css/themes/modern.css');
        }
""",
            """        if (theme.startsWith('modern')) {
            css_paths.unshift('/css/themes/modern.css');
        }
        if (theme === 'necgro') {
            css_paths = ['/css/themes/modern.css', '/ExtensionFile/NECGRO/Assets/necgro.css', '/ExtensionFile/NECGRO/Assets/necgro-user.css'];
            isDark = true;
        }
""",
            "theme === 'necgro'",
        ),
    ],
    "src/Pages/Login.cshtml": [
        (
            "Login to</span> SwarmUI:",
            "Login to</span> NECGRO:",
            "Login to</span> NECGRO:",
        ),
    ],
    "src/Pages/Register.cshtml": [
        (
            "Register an account for</span> SwarmUI:",
            "Register an account for</span> NECGRO:",
            "Register an account for</span> NECGRO:",
        ),
    ],
    "src/Pages/GoogleOAuthVerify.cshtml": [
        (
            "Register an account for</span> SwarmUI:",
            "Register an account for</span> NECGRO:",
            "Register an account for</span> NECGRO:",
        ),
    ],
}

_REQUIRED = {
    "src/Pages/Shared/_Layout.cshtml",
    "src/Pages/Install.cshtml",
    "src/wwwroot/js/installer.js",
}


def _looks_like_swarm(path):
    return os.path.isdir(os.path.join(path, "src", "wwwroot")) and os.path.isfile(
        os.path.join(path, "src", "Pages", "Shared", "_Layout.cshtml")
    )


def _resolve_swarm_root(swarm_root=None):
    candidates = []
    if swarm_root:
        candidates.append(swarm_root)
    candidates.append(os.getcwd())
    swarm_path = os.environ.get("SWARMPATH")
    if swarm_path:
        candidates.append(os.path.join(swarm_path, "SwarmUI"))
    candidates.extend(
        [
            "/content/drive/MyDrive/SwarmUI",
            "/content/SwarmUI",
            "/content/StableSwarmUI",
            "/content/drive/MyDrive/StableSwarmUI",
        ]
    )
    seen = set()
    for candidate in candidates:
        if not candidate or candidate in seen:
            continue
        seen.add(candidate)
        if _looks_like_swarm(candidate):
            return candidate
    raise FileNotFoundError("SwarmUI install not found. Clone it before applying the NECGRO skin.")


def _write(path, text):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def _patch_file(path, pairs):
    raw = path.read_bytes()
    text = raw.decode("utf-8")
    original = text
    for old, new, marker in pairs:
        if marker in text:
            continue
        if old not in text:
            print(f"NECGRO: skipped {path.name}; expected text was not found.")
            continue
        text = text.replace(old, new, 1)
    if text != original:
        path.write_bytes(text.encode("utf-8"))


def update_settings_text(text):
    newline = "\r\n" if "\r\n" in text else "\n"
    lines = text.splitlines()

    def indent_of(line):
        return len(line) - len(line.lstrip(" "))

    stack = []
    instance_idx = None
    theme_idx = None
    user_auth_idx = None
    default_user_idx = None
    for i, line in enumerate(lines):
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            continue
        ind = indent_of(line)
        while stack and stack[-1][0] >= ind:
            stack.pop()
        if ":" not in stripped:
            continue
        name, _, value = stripped.partition(":")
        name = name.strip()
        path = tuple(item[1] for item in stack)
        if name == "InstanceTitle" and path == ("UserAuthorization",):
            instance_idx = i
        if name == "Theme" and path == ("DefaultUser",):
            theme_idx = i
        if name == "UserAuthorization" and path == () and value.strip() == "":
            user_auth_idx = i
        if name == "DefaultUser" and path == () and value.strip() == "":
            default_user_idx = i
        if value.strip() == "":
            stack.append((ind, name))

    def set_line(idx, key, value):
        lines[idx] = (" " * indent_of(lines[idx])) + f"{key}: {value}"

    if instance_idx is not None:
        set_line(instance_idx, "InstanceTitle", "NECGRO")
    elif user_auth_idx is not None:
        lines.insert(user_auth_idx + 1, (" " * (indent_of(lines[user_auth_idx]) + 4)) + "InstanceTitle: NECGRO")
        if default_user_idx is not None and default_user_idx > user_auth_idx:
            default_user_idx += 1
        if theme_idx is not None and theme_idx > user_auth_idx:
            theme_idx += 1
    else:
        if lines and lines[-1].strip():
            lines.append("")
        lines.append("UserAuthorization:")
        lines.append("    InstanceTitle: NECGRO")

    if theme_idx is not None:
        set_line(theme_idx, "Theme", "necgro")
    elif default_user_idx is not None:
        lines.insert(default_user_idx + 1, (" " * (indent_of(lines[default_user_idx]) + 4)) + "Theme: necgro")
    else:
        if lines and lines[-1].strip():
            lines.append("")
        lines.append("DefaultUser:")
        lines.append("    Theme: necgro")

    body = newline.join(lines)
    if text.endswith(("\n", "\r\n")) or text == "":
        if not body.endswith("\n"):
            body += newline
    return body


def apply_necgro_skin(swarm_root=None):
    """Write the NECGRO extension, point settings at it, and rename visible SwarmUI chrome."""
    root = Path(_resolve_swarm_root(swarm_root))
    extension = root / "src" / "Extensions" / "NECGRO"
    _write(extension / "NECGRO.cs", NECGRO_CS)
    _write(extension / "NECGRO.csproj", NECGRO_CSPROJ)
    _write(extension / "Assets" / "necgro.css", NECGRO_THEME_CSS)
    _write(extension / "Assets" / "necgro-user.css", NECGRO_USER_CSS)
    _write(extension / "Assets" / "necgro-name.js", NECGRO_NAME_JS)

    for rel, pairs in _PATCHES.items():
        path = root / rel
        if not path.is_file():
            message = f"NECGRO: missing {rel}."
            if rel in _REQUIRED:
                raise FileNotFoundError(message)
            print(message)
            continue
        _patch_file(path, pairs)

    settings = root / "Data" / "Settings.fds"
    current = settings.read_text(encoding="utf-8") if settings.is_file() else ""
    _write(settings, update_settings_text(current))
    print(f"NECGRO skin applied in {root}")
    return str(root)

# Establish the environment and execute the linux launcher for SwarmUI
import os
import subprocess
import re

# Configure Swarm to use global pip packages rather than a local venv inside Colab
os.environ['SWARM_NO_VENV'] = 'true'

# Point to active installation path
swarm_dir = '/content/StableSwarmUI' if os.path.exists('/content/StableSwarmUI') else '/content/drive/MyDrive/SwarmUI'
os.chdir(swarm_dir)
print(f"Navigated to SwarmUI directory: {swarm_dir}")
apply_necgro_skin(swarm_dir)

cmd = ["bash", "./launch-linux.sh", "--launch_mode", "none", "--cloudflared-path", "cloudflared"]

print("[NECGRO] Starting the background server process...")
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Display output logs in real-time and look for the TryCloudflare link
url_captured = False
for line in process.stdout:
    print(line, end="")
    if not url_captured and "trycloudflare.com" in line:
        match = re.search(r'(https://[a-zA-Z0-9-]+\.trycloudflare\.com)', line)
        if match:
            tunnel_url = match.group(1)
            print("\n" + "="*60)
            print(f">>> [NECGRO]: {tunnel_url} <<<")
            print("="*60 + "\n")
            url_captured = True